<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

# 10a — Data Loading, QC & PCA

**Goal:** Load pseudobulk counts from `12_fragment_matrices/`, merge across species,
apply quality filters, and run PCA on **all shared regions** (pre-filtering) and
**filtered shared peaks** (post-filtering), plus **per-cell-type subsets**.

**Outputs saved to** `cross_species_consensus_v3/13_deseq2_R_pseudobulk/`:
- `pseudobulk/pb_data_shared.rds` — filtered shared-peak counts + metadata
- `pseudobulk/pb_data_union.rds`  — union-peak counts + metadata (for evolutionary branch testing)
- `plots/*All_Regions_PCA_*.pdf`  — PCA on all shared regions (before peak filtering)
- `plots/*Filtered_Peaks_PCA_*.pdf` — PCA on filtered shared peaks
- `plots/*_PCA_*.pdf`             — Per-cell-type PCA
- `plots/Count_Distributions_Per_Species_Log10.pdf`

**Input:** Parquet files from `cross_species_consensus_v3/12_fragment_matrices/{Species}/`

In [1]:
# =============================================================================
# Cell 1: Load Packages & Source Utility Functions
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(arrow)
  library(ggplot2)
  library(pheatmap)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})

# Source all reusable pipeline functions
source("../src/deseq2_utils.R")
message("All packages loaded & utility functions sourced.")

All packages loaded & utility functions sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration
# =============================================================================
BASE      <- "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
QUANT_DIR <- file.path(BASE, "cross_species_consensus_v3/12_fragment_matrices")
OUT_DIR   <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

# Filtering thresholds
MIN_SAMPLE_COUNTS <- 50000
MIN_CELLS         <- 100

# Aggregation settings
DO_AGGREGATE   <- TRUE
COLS_TO_MERGE  <- c("region")
COLS_TO_SUM    <- c("n_cells")

# Peak filtering
MIN_SAMPLES_PER_SPECIES <- 2
MIN_SPECIES_ACTIVE      <- 6
MAX_COUNT_THRESHOLD     <- 150

message("Configuration set. Output: ", OUT_DIR)

Configuration set. Output: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk



<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [3]:
# =============================================================================
# Cell 3: Load & Merge Pseudobulk Data
# =============================================================================
raw <- load_pseudobulk_data(QUANT_DIR, SPECIES)

# Inner join (shared peaks) — for global PCA and shared-peak DESeq2
shared <- merge_pseudobulk(raw$all_counts, raw$all_meta, join_type = "inner")
counts_merged <- shared$counts
meta_merged   <- shared$meta

Skipping Human: files not found at /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Human



Skipping Bonobo: files not found at /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Bonobo



Skipping Chimpanzee: files not found at /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Chimpanzee



Skipping Gorilla: files not found at /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Gorilla



Skipping Macaque: files not found at /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Macaque



Skipping Marmoset: files not found at /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Marmoset



Inner join: 0 shared peaks



ERROR: Error in counts_merged[, rownames(meta_merged)]: incorrect number of dimensions


In [ ]:
# =============================================================================
# Cell 4: Filter to Adult Samples & Aggregate
# =============================================================================
# Subset to Adults
meta_merged   <- meta_merged[meta_merged$age == "Adult", ]
counts_merged <- counts_merged[, rownames(meta_merged)]
message("Adult samples: ", ncol(counts_merged))

# Aggregate across regions (collapse biological replicates from different regions)
if (DO_AGGREGATE && length(COLS_TO_MERGE) > 0) {
  agg <- aggregate_pseudobulk(counts_merged, meta_merged,
                              merge_cols = COLS_TO_MERGE,
                              sum_cols   = COLS_TO_SUM)
  counts_merged <- agg$counts
  meta_merged   <- agg$meta
}

head(meta_merged)

In [ ]:
# =============================================================================
# Cell 5: QC — Count Distributions & Sample Filtering
# =============================================================================
meta_merged$total_counts <- colSums(counts_merged)

# Plot count distributions
plots_dir <- file.path(OUT_DIR, "plots")
dir.create(plots_dir, showWarnings = FALSE, recursive = TRUE)

plot_count_distribution(meta_merged,
                        file.path(plots_dir, "Count_Distributions_Per_Species_Log10.pdf"),
                        min_counts = MIN_SAMPLE_COUNTS,
                        min_cells  = MIN_CELLS)

# Apply quality filters
filt <- filter_samples(counts_merged, meta_merged,
                       min_counts = MIN_SAMPLE_COUNTS,
                       min_cells  = MIN_CELLS)
counts_filtered <- filt$counts
meta_filtered   <- filt$meta

# Setup factors
meta_filtered$species   <- factor(meta_filtered$species, levels = SPECIES)
meta_filtered$donor     <- factor(meta_filtered$donor)
meta_filtered$age       <- factor(meta_filtered$age)
meta_filtered$region    <- factor(meta_filtered$region)
meta_filtered$is_human  <- factor(ifelse(meta_filtered$species == "Human",
                                         "Human", "NonHuman"),
                                  levels = c("NonHuman", "Human"))
meta_filtered$cell_type <- as.factor(make.names(as.character(meta_filtered$cell_type)))

In [ ]:
# =============================================================================
# Cell 7: Global PCA — All Shared Peaks
# =============================================================================
vsd_global <- run_pca_suite(counts_filtered, meta_filtered, OUT_DIR,
                            prefix = "Shared_Peaks")

message("Global PCA on all shared peaks complete.")

In [ ]:
# =============================================================================
# Cell 6: Subset to Shared Cell Types & Species-Aware Peak Filtering
# =============================================================================
shared_ct <- subset_shared_celltypes(counts_filtered, meta_filtered)
counts_shared <- shared_ct$counts
meta_shared   <- shared_ct$meta

# Species-aware peak filtering
keep_peaks <- species_aware_peak_filter(
  counts_shared, meta_shared, SPECIES,
  min_samples_per_species = MIN_SAMPLES_PER_SPECIES,
  min_species_active      = MIN_SPECIES_ACTIVE,
  max_count_threshold     = MAX_COUNT_THRESHOLD
)
counts_shared <- counts_shared[keep_peaks, ]

message("Final shared-peak matrix: ", nrow(counts_shared), " peaks x ",
        ncol(counts_shared), " samples")

In [ ]:
# =============================================================================
# Cell 8: Global PCA — Filtered Shared Peaks
# =============================================================================
vsd_global <- run_pca_suite(counts_shared, meta_shared, OUT_DIR,
                            prefix = "Filtered_Peaks")

message("Global PCA on filtered shared peaks (", nrow(counts_shared),
        " peaks) complete.")

In [ ]:
# =============================================================================
# Cell 8: Per-Cell-Type PCA Subsets
# =============================================================================
# Run PCA for each cell type individually
cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  meta_ct   <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]

  # Skip if too few samples
  if (ncol(counts_ct) < 4 || length(unique(meta_ct$species)) < 2) {
    message("Skipping PCA for ", ct, ": too few samples.")
    next
  }

  # Light peak filter for this specific cell type
  keep_ct   <- rowSums(counts_ct >= 10) >= 2
  counts_ct <- counts_ct[keep_ct, ]

  tryCatch({
    run_pca_suite(counts_ct, meta_ct, OUT_DIR, prefix = ct)
  }, error = function(e) {
    message("  PCA failed for ", ct, ": ", e$message)
  })
}

message("\nPer-cell-type PCA complete.")

In [ ]:
# =============================================================================
# Cell 9: Save Intermediate Data for Downstream Notebooks
# =============================================================================
save_dir <- file.path(OUT_DIR, "pseudobulk")
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- Shared-peak data (for 10b shared-peak analysis) ----
saveRDS(list(counts = counts_shared, meta = meta_shared),
        file.path(save_dir, "pb_data_shared.rds"))

# ---- Union-peak data (for 10b contrast-peak analysis) ----
# Build a union-join matrix, then apply the SAME aggregation / filtering
# pipeline so that the column names match meta_shared (post-aggregation IDs).
union_raw <- merge_pseudobulk(raw$all_counts, raw$all_meta, join_type = "union")

# 1. Subset to Adults (same filter as Cell 4)
union_meta   <- union_raw$meta[union_raw$meta$age == "Adult", ]
union_counts <- union_raw$counts[, rownames(union_meta)]

# 2. Aggregate (same settings as Cell 4)
if (DO_AGGREGATE && length(COLS_TO_MERGE) > 0) {
  agg_union    <- aggregate_pseudobulk(union_counts, union_meta,
                                       merge_cols = COLS_TO_MERGE,
                                       sum_cols   = COLS_TO_SUM)
  union_counts <- agg_union$counts
  union_meta   <- agg_union$meta
}

# 3. Apply same quality filter (same thresholds as Cell 5)
union_meta$total_counts <- colSums(union_counts)
filt_union   <- filter_samples(union_counts, union_meta,
                               min_counts = MIN_SAMPLE_COUNTS,
                               min_cells  = MIN_CELLS)
union_counts <- filt_union$counts
union_meta   <- filt_union$meta

# 4. Keep only the samples that survived shared-cell-type filtering
valid_samples     <- intersect(rownames(meta_shared), colnames(union_counts))
counts_union_filt <- union_counts[, valid_samples, drop = FALSE]
meta_union_filt   <- meta_shared[valid_samples, ]

message("Union matrix after full pipeline: ", nrow(counts_union_filt),
        " peaks x ", ncol(counts_union_filt), " samples")

saveRDS(list(counts = counts_union_filt, meta = meta_union_filt),
        file.path(save_dir, "pb_data_union.rds"))

# ---- Global VST for heatmaps in 10b ----
saveRDS(vsd_global, file.path(save_dir, "vsd_global_shared.rds"))

message("\n=== All intermediate data saved ===")
message("  pb_data_shared.rds  : ", nrow(counts_shared), " peaks x ",
        ncol(counts_shared), " samples")
message("  pb_data_union.rds   : ", nrow(counts_union_filt), " peaks x ",
        ncol(counts_union_filt), " samples")
message("  vsd_global_shared.rds")

## Per-Region Data (No Region Aggregation)

Save pseudobulk data with the **region column preserved** (Duodenum and Colon kept separate).
This is required for `run_deseq2_per_region()` in notebook `10b_region_specific`.

Key differences vs the main pipeline:
- Region NOT aggregated — each sample = donor × cell_type × region
- Adults only (same filter)
- Lower QC thresholds: counts ≥ 30,000 and cells ≥ 50 (per-region pseudobulks have fewer cells)
- Cell type names cleaned identically to main pipeline (`make.names`)
- Shared peak IDs are the same 71,663 filtered peaks from above

In [ ]:
# =============================================================================
# Cell 10: Save Region-Preserved Data for Per-Region DESeq2
# =============================================================================
# These thresholds are lower because per-region pseudobulks have fewer cells
# than region-merged ones.
MIN_COUNTS_REGION <- 30000
MIN_CELLS_REGION  <- 50

# ---- Shared peaks, per region ----
# 'shared$meta' still holds the full pre-aggregation metadata (from Cell 3)
# with region, donor, cell_type, species, age all intact.
meta_preg <- shared$meta[shared$meta$age == "Adult", ]
counts_preg <- shared$counts[, rownames(meta_preg)]

# Apply region-level QC
meta_preg$total_counts <- colSums(counts_preg)
keep_preg <- meta_preg$total_counts >= MIN_COUNTS_REGION
if ("n_cells" %in% colnames(meta_preg)) {
  keep_preg <- keep_preg & (meta_preg$n_cells >= MIN_CELLS_REGION)
}
meta_preg   <- meta_preg[keep_preg, ]
counts_preg <- counts_preg[, rownames(meta_preg)]
message("Per-region QC: kept ", ncol(counts_preg), " samples")
print(table(meta_preg$region, meta_preg$species))

# Apply same cell type name cleaning as main pipeline
meta_preg$cell_type <- as.factor(make.names(as.character(meta_preg$cell_type)))
meta_preg$species   <- factor(meta_preg$species, levels = SPECIES)
meta_preg$region    <- factor(meta_preg$region)

# Restrict to the same 71,663 shared + species-aware filtered peaks
shared_peak_ids <- rownames(counts_shared)   # from Cell 6 above
counts_preg_shared <- counts_preg[shared_peak_ids, , drop = FALSE]
message("Per-region shared-peak matrix: ", nrow(counts_preg_shared),
        " peaks x ", ncol(counts_preg_shared), " samples")

saveRDS(list(counts = counts_preg_shared, meta = meta_preg),
        file.path(save_dir, "pb_data_shared_per_region.rds"))

# ---- Union peaks, per region ----
# union_raw was built in Cell 9 above; reuse its inner structure
meta_union_preg   <- union_raw$meta[union_raw$meta$age == "Adult", ]
counts_union_preg <- union_raw$counts[, rownames(meta_union_preg)]

meta_union_preg$total_counts <- colSums(counts_union_preg)
keep_up <- meta_union_preg$total_counts >= MIN_COUNTS_REGION
if ("n_cells" %in% colnames(meta_union_preg)) {
  keep_up <- keep_up & (meta_union_preg$n_cells >= MIN_CELLS_REGION)
}
meta_union_preg   <- meta_union_preg[keep_up, ]
counts_union_preg <- counts_union_preg[, rownames(meta_union_preg)]

# Apply same cell type + species factor cleaning
meta_union_preg$cell_type <- as.factor(make.names(as.character(meta_union_preg$cell_type)))
meta_union_preg$species   <- factor(meta_union_preg$species, levels = SPECIES)
meta_union_preg$region    <- factor(meta_union_preg$region)

message("Per-region union-peak matrix: ", nrow(counts_union_preg),
        " peaks x ", ncol(counts_union_preg), " samples")

saveRDS(list(counts = counts_union_preg, meta = meta_union_preg),
        file.path(save_dir, "pb_data_union_per_region.rds"))

message("\n=== Per-region data saved ===")
message("  pb_data_shared_per_region.rds : ", nrow(counts_preg_shared), " peaks x ",
        ncol(counts_preg_shared), " samples")
message("  pb_data_union_per_region.rds  : ", nrow(counts_union_preg), " peaks x ",
        ncol(counts_union_preg), " samples")
message("  Regions: ", paste(levels(meta_preg$region), collapse = ", "))